# W1 浏览器真实 FPGA 闭环
使用原 A4 的 bit/HWH；无 HDMI。上传本 Notebook 和 `browser_a4_w1_r1.zip` 到一个新建文件夹。
每个代码框只执行一次，按顺序等待 In[数字] 且无红色报错。不要 Run All。
本阶段阈值固定28，可切换两张图/256或512/四个真实PL模式。主机通过不是新板测通过。

## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('browser_a4_w1_r1.zip')
expected_bundle_sha256 = 'e25c3cfeeb6379338e9f590ab4e1d0df8d02d60b9386d6136091188a81feb34d'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('browser_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'browser_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 100 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'browser_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'browser_a4_w1_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: original A4 and browser application bytes checked; FPGA untouched')


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [ ]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from pynq_app.launch import prepare, start, export_session, finish
browser_context = prepare(browser_package_root, port=8090)


## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [ ]:
browser_start_status = await start(browser_context)
print(browser_start_status)


## 4. 在 Windows 浏览器操作
打开上一框打印的 http://板卡IP:8090/。若IP仍为192.168.2.99，地址为 http://192.168.2.99:8090/ 。
依次：im0001 /512/模式3运行；im0002 /512/模式3运行；im0002 /512/模式1运行；im0001 /256/模式3运行。
每次等待同帧四图出现、任务ID更新及收发字节/帧计数增加。运行旁路时只有原图是正常的。
然后选 im0001 /512/模式3，点击“连续运行20帧”，完成后截图界面。只在前台可见时取得连续展示FPS。
阈值不能调是W1真实能力限制；透明度只改变叠加显示，不增加PL任务。
此时可在网页“导出当前任务/本次会话”。不要关闭此Notebook、重启内核或运行其他Overlay。
网页停止后续请求不会取消已经发送的DMA；等待板端IDLE后再执行下框。

In [ ]:
print(browser_context['manager'].status())


## 5. 打包本次真实记录
须无在途/待执行任务。应打印 BROWSER_RECORDS_READY。完整会话包含每任务原packed NPY、PNG、JSON、CSV及浏览器绘制观测；尚待项目独立复核，不自动宣称验收通过。

In [ ]:
import shutil
browser_export_path = await export_session(browser_context)
shutil.copyfile(browser_export_path, Path.cwd() / 'browser_session_bundle.zip')
print('DOWNLOAD: browser_session_bundle.zip in this notebook folder')


## 6. 安全结束并下载
仅READY/IDLE且无在途或故障缓冲时关闭HTTP服务。FAULT时本框拒绝关闭，保留内核和原记录并反馈。
完成后Ctrl+S；File → Download as → Notebook下载执行记录。回文件列表下载 `browser_session_bundle.zip`。反馈这两个文件和浏览器截图。

In [ ]:
await finish(browser_context)
